In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# --- 1. SETUP PATHS ---
# Define the locations of your source files and where the output should go
MAIN_DATA_FILE = 'C:/Users/511232/Desktop/DSS/MERGING GOOGLESHEETS QUESTIONNAIRES/codes/arabic_questionnaires.xlsx'
CRITERIA_FILE = 'C:/Users/511232/Desktop/criterias.xlsx'
OUTPUT_FILE = 'simplified_availability_results.xlsx'

# --- 2. LOAD & INITIAL CLEANING ---
# Load the raw data from Excel
main_df = pd.read_excel(MAIN_DATA_FILE)
criteria_df = pd.read_excel(CRITERIA_FILE)

# Remove any rows where the 'Value' column is empty/NaN as per your requirement
main_df = main_df.dropna(subset=['القيمة'])

# Keep only the specific columns you need for the logic
cols_to_keep = ['المؤشر', 'الدولة', 'السنة', 'المواطنة', 'الجنس']
main_df = main_df[cols_to_keep]

In [ ]:
# Prepare the criteria lookup table; clean up empty indicator names first
criteria_df = criteria_df.dropna(subset=['Indicator_Ar'])
# We keep this as a DataFrame to merge it easily later
criteria_df = criteria_df[['Indicator_Ar', 'criteria']]

In [ ]:
# --- 3. CREATE BRACKETS & VALID INDICATOR ---
# Create a list of conditions for the 5-year year brackets
conditions = [
    (main_df['السنة'] >= 2010) & (main_df['السنة'] <= 2014),
    (main_df['السنة'] >= 2015) & (main_df['السنة'] <= 2019),
    (main_df['السنة'] >= 2020) & (main_df['السنة'] <= 2024),
    (main_df['السنة'] >= 2025)
]

# Define the labels for these brackets
choices = ['2010-2014', '2015-2019', '2020-2024', '2025+']

# Apply the labels to a new column called 'year_bracket'
main_df['year_bracket'] = np.select(conditions, choices, default='Other')
# Create a 'valid' column with 1 to represent a data point exists
main_df['valid'] = 1

In [ ]:
# --- 4. GENERAL AVAILABILITY ---
# Remove nationality and sex details then drop duplicates to collapse the data
# This ensures we count unique years per indicator/country regardless of sub-categories
df_gen = main_df[['المؤشر', 'الدولة', 'السنة', 'year_bracket', 'valid']].drop_duplicates()

# Group by Indicator, Country, and Bracket to see how many data points are in each 5-year window
gen_grouped = df_gen.groupby(['المؤشر', 'الدولة', 'year_bracket'])['valid'].sum().reset_index()

# Merge with the criteria file to see the required count for each indicator
gen_merged = pd.merge(gen_grouped, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')

# Check if the sum of valid points in that bracket meets or exceeds the criteria
gen_merged['met_criteria'] = (gen_merged['valid'] >= gen_merged['criteria']).astype(int)

# Count how many brackets for each Indicator/Country met the criteria
# Assuming you want all 3 main brackets (2010-2024) to be successful
final_gen = gen_merged.groupby(['المؤشر', 'الدولة'])['met_criteria'].sum().reset_index()
final_gen['general_availability'] = (final_gen['met_criteria'] >= 3).astype(int)
final_gen = final_gen[['المؤشر', 'الدولة', 'general_availability']]

In [ ]:
# --- 5. NATIONALITY AVAILABILITY ---
# 1. Filter for valid nationality entries
invalid_entries = ['Not applicable', 'غير متاح', 'Total']
df_nat = main_df[main_df['المواطنة'].notna() & ~main_df['المواطنة'].isin(invalid_entries)].copy()

# 2. First collapse: Group by Year and take the max of 'valid'
# This ensures that for a single year, if 'National' OR 'Non-national' exists, 'valid' is 1 (not 2)
df_nat_yearly = df_nat.groupby(['المؤشر', 'الدولة', 'السنة', 'year_bracket'])['valid'].max().reset_index()

# 3. Second collapse: Group by the 5-year Bracket and sum the valid years
# Now we are counting how many unique years in that bracket had nationality data
nat_grouped = df_nat_yearly.groupby(['المؤشر', 'الدولة', 'year_bracket'])['valid'].sum().reset_index()

# 4. Merge with criteria to see how many years are required for this indicator
nat_merged = pd.merge(nat_grouped, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')

# 5. Check if the number of valid years in the bracket meets the criteria
nat_merged['met_criteria'] = (nat_merged['valid'] >= nat_merged['criteria']).astype(int)

# 6. Final summary: Check if all 3 required brackets (2010-2024) met the criteria
final_nat = nat_merged.groupby(['المؤشر', 'الدولة'])['met_criteria'].sum().reset_index()
final_nat['nationality_availability'] = (final_nat['met_criteria'] >= 3).astype(int)
final_nat = final_nat[['المؤشر', 'الدولة', 'nationality_availability']]


# --- 6. SEX AVAILABILITY ---
# 1. Filter for valid sex entries
df_sex = main_df[main_df['الجنس'].notna() & ~main_df['الجنس'].isin(invalid_entries)].copy()

# 2. First collapse: Ensure a year is marked '1' if Male OR Female exists
df_sex_yearly = df_sex.groupby(['المؤشر', 'الدولة', 'السنة', 'year_bracket'])['valid'].max().reset_index()

# 3. Second collapse: Sum the unique valid years within each 5-year bracket
sex_grouped = df_sex_yearly.groupby(['المؤشر', 'الدولة', 'year_bracket'])['valid'].sum().reset_index()

# 4. Merge with criteria
sex_merged = pd.merge(sex_grouped, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')

# 5. Check if the bracket meets the criteria
sex_merged['met_criteria'] = (sex_merged['valid'] >= sex_merged['criteria']).astype(int)

# 6. Final summary across brackets
final_sex = sex_merged.groupby(['المؤشر', 'الدولة'])['met_criteria'].sum().reset_index()
final_sex['sex_availability'] = (final_sex['met_criteria'] >= 3).astype(int)
final_sex = final_sex[['المؤشر', 'الدولة', 'sex_availability']]

In [ ]:
# --- 7. FINAL MERGE & EXPORT ---
# Combine all three availability results into one table
results = pd.merge(final_gen, final_nat, on=['المؤشر', 'الدولة'], how='left')
results = pd.merge(results, final_sex, on=['المؤشر', 'الدولة'], how='left')

# Fill missing values with 0 (meaning no data was found for that category)
results = results.fillna(0).astype({'nationality_availability': int, 'sex_availability': int})

# Save the final report
results.to_excel(OUTPUT_FILE, index=False)
print(f"Report generated: {OUTPUT_FILE}")